# 2. Cell-type classification: markers vs a reference

Two ways to name cells: from
**marker genes** (reference-free) and from a **single-cell reference** (here Lake / KPMP
kidney snRNA-seq) via a trained classifier. We run both and compare.

### Setup

In [ ]:
import sys; sys.path.insert(0, '../scripts')
import numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt
import course_utils as cu
sc.settings.verbosity = 1
plt.rcParams.update({'figure.facecolor': 'white', 'figure.dpi': 110})

## 1. Load and normalise

In [ ]:
adata = cu.load_or_build_qc()
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata); sc.pp.log1p(adata)   # within the 480-gene panel
print(adata.shape)

## 2. Marker-based labels (reference-free)

> **Method note.** The simplest annotation:
score each marker set per cell and take the top-scoring lineage. Transparent and needs no
reference, but coarse, and it cannot resolve types whose markers are off the panel.

In [ ]:
# (A) marker-based, reference-free: per-cell argmax of marker scores
markers = cu.load_markers()
markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}
markers = {k: v for k, v in markers.items() if v}
for ct, genes in markers.items():
    sc.tl.score_genes(adata, genes, score_name=f's::{ct}')
S = adata.obs[[f's::{ct}' for ct in markers]]
adata.obs['celltype_marker'] = pd.Categorical(
    S.values.argmax(1)).rename_categories({i: list(markers)[i] for i in range(len(markers))})
adata.obs['celltype_marker'].value_counts().head(12)

## 3. Reference-based labels (supervised)

> **Method note: training on a reference.**
We train a regularised logistic-regression classifier (the idea behind CellTypist) on the
Lake reference using its `subclass.l1` labels, restricted to the genes shared with the
panel, and apply it to the Xenium cells with a confidence (max class probability).
**Caveat:** the reference is dissociated snRNA-seq and Xenium is imaging-based on a small
panel, so there is a platform gap; the classifier ignores spatial context and treats every
cell independently. scANVI or CellTypist with its own model are production alternatives,
and notebook 3 integrates the platforms to reduce the gap.

In [ ]:
# (B) reference-based: a logistic-regression classifier trained on Lake/KPMP snRNA-seq
ref = cu.load_lake_reference(shared_with=adata.var_names)   # subset to shared panel genes
# the reference X is already log-normalised; a per-gene scaler (below) handles the
# remaining scale difference between snRNA-seq and the Xenium panel
label = 'subclass.l1'
# subsample the reference for a fast, balanced fit (more cells = better in production)
rng = np.random.default_rng(0)
idx = (ref.obs.groupby(label, observed=True).apply(
    lambda d: d.sample(min(len(d), 2500), random_state=0)).index.get_level_values(1))
ref_s = ref[ref.obs_names.isin(idx)].copy()
print('training cells:', ref_s.n_obs, '| classes:', ref_s.obs[label].nunique())

In [ ]:
from sklearn.preprocessing import StandardScaler
genes = list(ref_s.var_names)                       # shared genes, common order
Xtr = ref_s[:, genes].X; Xtr = Xtr.toarray() if hasattr(Xtr, 'toarray') else np.asarray(Xtr)
ytr = ref_s.obs[label].astype(str).values
scaler = StandardScaler().fit(Xtr)                  # standardise features for the classifier
Xte = adata[:, genes].X; Xte = Xte.toarray() if hasattr(Xte, 'toarray') else np.asarray(Xte)
print('train on:', Xtr.shape, '| predict on:', Xte.shape)

> **Exercise (ML).** Train a classifier on the scaled reference features
(`scaler.transform(Xtr)`, labels `ytr`) and use it to predict `adata.obs['celltype_ref']`
plus a per-cell confidence (`adata.obs['celltype_ref_conf']`) for the Xenium cells. A
regularised **logistic regression** is a strong, fast baseline, start there. (You will
compare other algorithms in the second ML exercise below.)

In [ ]:
# your code here


## 4. Compare the two annotations

> **Method note.** The label vocabularies differ
(verbose marker names vs `subclass.l1` codes), so we read the cross-tabulation: each
marker lineage should map mostly to the matching classifier class (e.g. proximal-tubule
markers -> `PT`). Disagreements concentrate in similar types (PT vs TAL) and where the
panel is thin.

In [ ]:
# how the two annotations relate (rows: marker argmax, cols: classifier)
ct = pd.crosstab(adata.obs['celltype_marker'], adata.obs['celltype_ref'], normalize='index').round(2)
ct

In [ ]:
for col in ['celltype_ref', 'celltype_marker']:
    sub = adata[adata.obs['sample'] == 'X37']
    sq.pl.spatial_scatter(sub, color=col, shape=None, size=8, figsize=(7, 7))

> **Exercise.** Find the cells where the two methods disagree and check the classifier's
confidence there. Are disagreements concentrated in low-confidence cells or in particular
types?

In [ ]:
# your code here


## 5. Save the annotation

In [ ]:
# persist the annotation for notebooks 3-5 (working label = the reference classifier)
adata.obs['celltype'] = adata.obs['celltype_ref']
from pathlib import Path; Path('results').mkdir(exist_ok=True)
adata.write('results/slide_0011695_annotated.h5ad')
print('saved results/slide_0011695_annotated.h5ad')

> **Exercise (ML).** We used logistic regression. Train two other classifiers, a
**random forest** and a **kNN**, on the same reference features and compare their
cross-validated accuracy on the reference. Which generalises best, and does a more flexible
model actually help on this 480-gene panel?

In [ ]:
# your code here


### Recap

Marker scoring is transparent but coarse; a reference classifier is finer
but inherits a platform gap and ignores space. Use them together, and treat low-confidence
calls with caution. Next: integrate all samples to clean up batch and platform effects.